In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [13]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,\
                            precision_score,recall_score,\
                            f1_score,confusion_matrix

from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier,OneVsOneClassifier
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
warnings.filterwarnings("ignore")

In [14]:
df = pd.read_csv("data/spotify-tracks.csv",index_col=0)


In [15]:
bins = [0, 30, 60, 80, 100]
labels = ["Poco popular", "Moderadamente popular", "Popular", "Muy popular"]
df["popularity_class"] = pd.cut(df["popularity"], bins=bins, labels=labels)

In [16]:
df.dropna(inplace=True)


In [17]:
df['artists'] = df['artists'].apply(lambda x: str(x).replace(';',' ').strip())
df['duration_ms']=df['duration_ms']/60000

In [18]:
from sklearn.base import clone
from scipy import stats
from itertools import combinations




class MultiClassClassifier:
    def __init__(self, models, df, target_column='popularity_class',
                 numeric_features=None, categorical_features=None,
                 apply_pca=False, n_components=None,
                 feature_selection=False, k_features=None):
        """
        Inicializa el clasificador multiclase con parámetros fijos.

        Parámetros:
            models (dict): Diccionario de modelos a entrenar (con wrappers OVO/OVR ya configurados)
            df (DataFrame): DataFrame completo
            target_column (str): Nombre de la columna objetivo
            numeric_features (list): Lista de características numéricas
            categorical_features (list): Lista de características categóricas
            apply_pca (bool): Si aplicar PCA
            n_components (int): Número de componentes para PCA
            feature_selection (bool): Si aplicar selección de características
            k_features (int): Número de características a seleccionar
        """
        self.df = df
        self.target_column = target_column
        self.numeric_features = numeric_features if numeric_features else []
        self.categorical_features = categorical_features if categorical_features else []
        self.apply_pca = apply_pca
        self.n_components = n_components
        self.feature_selection = feature_selection
        self.k_features = k_features
        self.models = models
        self.preprocessor = None
        self.pipeline = None
        self.split_results = {}
        self.confusion_matrices = {}
        self.statistical_analysis = {}
        self.class_labels = None

    def _create_pipeline(self, model):
        """Crea el pipeline completo para un modelo."""

        numeric_transformer = StandardScaler()
        categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, self.numeric_features),
                ('cat', categorical_transformer, self.categorical_features)
            ],
            remainder='passthrough'
        )

        steps = [('preprocessor', preprocessor)]

        if self.feature_selection and self.k_features:
            steps.append(('feature_selection', SelectKBest(score_func=f_classif, k=self.k_features)))

        if self.apply_pca and self.n_components:
            steps.append(('pca', PCA(n_components=self.n_components)))

        steps.append(('clf', model))

        return Pipeline(steps)

    def train(self, X, y, test_size=0.2, n_repeats=5, random_state=None):
        """Entrena los modelos con parámetros fijos y evalúa múltiples veces."""
        self.class_labels = sorted(y.unique())
        self.split_results = {model_name: {
            'accuracy': [],
            'f1_macro': [],
            'f1_weighted': [],
            'precision_macro': [],
            'recall_macro': []
        } for model_name in self.models}

        self.confusion_matrices = {model_name: [] for model_name in self.models}

        for i in range(n_repeats):
            current_seed = random_state + i if random_state is not None else None
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=test_size, stratify=y, random_state=current_seed
            )
            for model_name, model in self.models.items():
                try:
                    pipeline = self._create_pipeline(clone(model))
                    print(f'train: {model_name}')
                    pipeline.fit(X_train, y_train)
                    print("done")
                    y_pred = pipeline.predict(X_test)


                    self.split_results[model_name]['accuracy'].append(accuracy_score(y_test, y_pred))
                    self.split_results[model_name]['f1_macro'].append(f1_score(y_test, y_pred, average='macro', zero_division=0))
                    self.split_results[model_name]['f1_weighted'].append(f1_score(y_test, y_pred, average='weighted', zero_division=0))
                    self.split_results[model_name]['precision_macro'].append(precision_score(y_test, y_pred, average='macro', zero_division=0))
                    self.split_results[model_name]['recall_macro'].append(recall_score(y_test, y_pred, average='macro', zero_division=0))

                    cm = confusion_matrix(y_test, y_pred, labels=self.class_labels)
                    self.confusion_matrices[model_name].append(cm)

                    print(f"Repetición {i+1}, Modelo {model_name} completado.")
                except Exception as e:
                    print(f"Error entrenando {model_name} en repetición {i+1}: {str(e)}")

            print(f"Repetición {i+1}/{n_repeats} completada para todos los modelos.")

    def plot_results(self, metrics=['f1_macro', 'accuracy', 'precision_macro', 'recall_macro']):
        """
        Crea boxplots para comparar la distribución de varias métricas.
        """
        num_metrics = len(metrics)
        plt.figure(figsize=(15, 5 * num_metrics))

        for i, metric in enumerate(metrics):
            plot_data = []
            for model_name, results in self.split_results.items():
                if metric in results:
                    for score in results[metric]:
                        plot_data.append({'model': model_name, 'score': score})

            df_plot = pd.DataFrame(plot_data)

            plt.subplot(num_metrics, 1, i + 1)
            sns.boxplot(x='model', y='score', data=df_plot, palette='viridis')

            metric_name = metric.replace('_', ' ').capitalize()
            plt.title(f'Distribución de {metric_name} en {len(self.split_results[list(self.models.keys())[0]][metric])} Repeticiones', fontsize=14)
            plt.ylabel(metric_name, fontsize=12)
            plt.xlabel('')
            plt.xticks(rotation=45, ha='right')
            plt.grid(axis='y', linestyle='--', alpha=0.7)

        plt.xlabel('Modelo', fontsize=12)
        plt.tight_layout()
        plt.show()

    def plot_mean_confusion_matrices(self, normalize=None):
        """
        Calcula y grafica la matriz de confusión promedio para cada modelo.
        """
        model_names = list(self.models.keys())
        num_models = len(model_names)

        cols = 2
        rows = (num_models + 1) // cols

        fig, axes = plt.subplots(rows, cols, figsize=(15, 6 * rows))
        if num_models == 1:  
            axes = np.array([axes])
        axes = axes.flatten()

        for i, model_name in enumerate(model_names):
            sum_cm = np.sum(self.confusion_matrices[model_name], axis=0)

            df_cm = pd.DataFrame(sum_cm, index=self.class_labels, columns=self.class_labels)

            sns.heatmap(df_cm, annot=True,
                       cmap='Bl*ues', ax=axes[i], cbar=False)
            axes[i].set_title(f'{model_name}\nMatriz de Confusión Promedio', fontsize=12)
            axes[i].set_ylabel('Clase Real', fontsize=10)
            axes[i].set_xlabel('Clase Predicha', fontsize=10)
            axes[i].tick_params(axis='both', which='major', labelsize=8)

        for j in range(i + 1, len(axes)):
            fig.delaxes(axes[j])

        plt.tight_layout()
        plt.show()

    def perform_statistical_analysis(self, metric='f1_macro', alpha=0.05):
        """
        Realiza un Test de Friedman seguido de un Test post-hoc de Nemenyi
        y visualiza los resultados con un Diagrama de Diferencia Crítica.
        """
        print(f"\n--- Análisis Estadístico Avanzado (Métrica: {metric}) ---")
        
        df_scores = pd.DataFrame({name: results[metric] for name, results in self.split_results.items()})
        
        stat, p_friedman = stats.friedmanchisquare(*df_scores.values.T)
        print(f"\n1. Test de Friedman (Global):")
        print(f"   - Estadística = {stat:.4f}")
        print(f"   - p-valor = {p_friedman:.6f}")
        
        if p_friedman >= alpha:
            print("\nConclusión: No hay una diferencia estadísticamente significativa entre el rendimiento de los modelos.")
            return None
        
        print(f"\nConclusión: Se rechaza la hipótesis nula. Hay al menos un modelo con rendimiento diferente.")
        print("Procediendo con el análisis post-hoc...")
        
        self._plot_critical_difference_diagram(df_scores, alpha=alpha)
        
        return df_scores

    def _plot_critical_difference_diagram(self, df_scores, alpha=0.05):
        """Calcula y grafica el Diagrama de Diferencia Crítica (CD Diagram)."""
        k = len(df_scores.columns) 
        n = len(df_scores)         

        ranks = df_scores.rank(axis=1, method='average', ascending=False)
        avg_ranks = ranks.mean().sort_values()
        q_alpha_map = {2:1.960, 3:2.343, 4:2.569, 5:2.728, 6:2.850, 7:2.949, 8:3.031, 9:3.102, 10:3.164}
        q_alpha = q_alpha_map.get(k, 3.164) 
        
        cd = q_alpha * np.sqrt(k * (k + 1) / (6 * n))
        
        plt.figure(figsize=(12, max(4, k * 0.5)))
        plt.title(f'Diagrama de Diferencia Crítica (Test de Nemenyi, alpha={alpha})', fontsize=16)
        
        for i, model_name in enumerate(avg_ranks.index):
            rank = avg_ranks[model_name]
            plt.plot([rank, rank], [i - 0.1, i + 0.1], 'k')
            plt.text(rank, i + 0.2, f'{model_name} ({rank:.2f})', ha='center', va='bottom')

        for i in range(k):
            for j in range(i + 1, k):
                model1 = avg_ranks.index[i]
                model2 = avg_ranks.index[j]
                rank1 = avg_ranks[model1]
                rank2 = avg_ranks[model2]
                
                if abs(rank1 - rank2) <= cd:
                    y_pos = -0.5 - (j - i) * 0.1 
                    plt.plot([rank1, rank2], [y_pos, y_pos], 'k', linewidth=3)

        plt.yticks([]) 
        plt.xlabel('Ranking Promedio (Menor es Mejor)', fontsize=12)
        plt.gca().invert_xaxis() 
        plt.grid(axis='x', linestyle='--', alpha=0.7)
        plt.show()
        self.plot_performance_over_repetitions()
        
    def plot_performance_over_repetitions(self, metric='f1_macro'):
        """
        Crea un gráfico de líneas para mostrar el rendimiento de cada modelo en cada repetición.

        Esto ayuda a visualizar la estabilidad y consistencia de los modelos.
        """
        plt.figure(figsize=(16, 8))
        
        df_results = pd.DataFrame({
            name: results[metric] for name, results in self.split_results.items()
        })

        sns.lineplot(data=df_results, dashes=False, palette='viridis', linewidth=2.5, marker='o', markersize=8)

        metric_name = metric.replace('_', ' ').capitalize()
        n_reps = len(df_results)
        
        plt.title(f'Evolución del Rendimiento ({metric_name}) a través de las Repeticiones', fontsize=16)
        plt.xlabel('Número de Repetición', fontsize=12)
        plt.ylabel(f'Puntuación de {metric_name}', fontsize=12)
        
        plt.xticks(ticks=range(0, n_reps, max(1, n_reps // 20)), labels=range(1, n_reps + 1, max(1, n_reps // 20)))
        
        plt.legend(title='Modelos', fontsize=10)
        plt.grid(True, which='both', linestyle='--', linewidth=0.5)
        plt.tight_layout()
        plt.show()
    def get_best_model(self, metric='f1_macro'):
        """Devuelve el mejor modelo según la métrica especificada."""
        if not self.split_results:
            raise ValueError("Los modelos no han sido entrenados aún. Llama a train() primero.")

        avg_scores = {
            model: np.mean(scores[metric])
            for model, scores in self.split_results.items()
        }
        best_model = max(avg_scores.items(), key=lambda x: x[1])

        print(f"\nMejor modelo según {metric}:")
        print(f"Modelo: {best_model[0]}")
        print(f"Puntuación media: {best_model[1]:.4f}")

        return best_model[0]

In [19]:
X = df.drop(columns=['track_id','album_name','artists','track_name','popularity_class','popularity','track_genre'])
y = df['popularity_class']
df.shape

(97980, 21)

In [20]:
from sklearn.naive_bayes import GaussianNB


models = {
    'LogisticRegression_OVR': OneVsRestClassifier(
        LogisticRegression(
            penalty='elasticnet',
            solver='saga',
            max_iter=1000,
            C=1.0,
            l1_ratio=0.5,
            class_weight='balanced',
            random_state=42
        )
    ),
    'LogisticRegression_OVO': OneVsOneClassifier(
        LogisticRegression(
            penalty='elasticnet',
            solver='saga',
            max_iter=1000,
            C=1.0,
            l1_ratio=0.5,
            class_weight='balanced',
            random_state=42
        )
    ),

    'RandomForest_OVO': OneVsOneClassifier(
        RandomForestClassifier(
            n_estimators=100,
            max_depth=20,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features='sqrt',
            class_weight='balanced_subsample',
            bootstrap=True,
            oob_score=True,
            random_state=42,
            n_jobs=1
        )
    ),
    'RandomForest_OVR': OneVsRestClassifier(
        RandomForestClassifier(
            n_estimators=200,
            max_depth=20,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features='sqrt',
            class_weight='balanced_subsample',
            bootstrap=True,
            oob_score=True,
            random_state=42,
            n_jobs=1
        )
    ),

    'XGBoost_OVR': OneVsRestClassifier(
        XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            gamma=0.1,
            min_child_weight=3,
            reg_alpha=0.1,
            reg_lambda=1.0,
            objective='binary:logistic',
            use_label_encoder=False,
            eval_metric='logloss',
            random_state=42,
            n_jobs=1,
            tree_method='hist' 
        )
    ),
    'XGBoost_OVO': OneVsOneClassifier(
        XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            gamma=0.1,
            min_child_weight=3,
            reg_alpha=0.1,
            reg_lambda=1.0,
            objective='binary:logistic',
            use_label_encoder=False,
            eval_metric='logloss',
            random_state=42,
            n_jobs=1,
            tree_method='hist'
        )
    ),
    'KNN_OVR': OneVsRestClassifier(
        KNeighborsClassifier(
            n_neighbors=5,
            weights='distance',
            algorithm='auto',
            leaf_size=30,
            p=2,
            metric='minkowski',
            n_jobs=1
        )
    ),
    'DecisionTree_OVO': OneVsOneClassifier(
        DecisionTreeClassifier(
            criterion='gini',
            splitter='best',
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features=None,
            class_weight='balanced',
            random_state=42
        )
    ),'DecisionTree_OVR': OneVsRestClassifier(
        DecisionTreeClassifier(
            criterion='gini',
            splitter='best',
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features=None,
            class_weight='balanced',
            random_state=42
        )
    ),
    'NaiveBayes_OVR': OneVsRestClassifier(
        GaussianNB(
            priors=None,
            var_smoothing=1e-9
        )
    ),'NaiveBayes_OVO': OneVsOneClassifier(
        GaussianNB(
            priors=None,
            var_smoothing=1e-9
        )
    ),'GradientBoosting_OVR': OneVsRestClassifier(
        GradientBoostingClassifier(
            n_estimators=300,
            learning_rate=0.1,
            max_depth=6,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features='sqrt',
            subsample=0.8,
            random_state=42
        )
    ),
    'GradientBoosting_OVO': OneVsOneClassifier(
        GradientBoostingClassifier(
            n_estimators=300,
            learning_rate=0.1,
            max_depth=6,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features='sqrt',
            subsample=0.8,
            random_state=42
        )
    )
}

numeric_features = ['danceability', 'energy', 'loudness', 'speechiness',
                   'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
categorical_features = ['track_genre']

In [ ]:
classifier = MultiClassClassifier(
    models=models,
    df=df,
    numeric_features=numeric_features)

classifier.train(X, y, n_repeats=30)


classifier.plot_results(metrics=['f1_macro', 'f1_weighted', 'accuracy', 'precision_macro', 'recall_macro'])
classifier.plot_mean_confusion_matrices(normalize='true')

stats_results = classifier.perform_statistical_analysis(metric='f1_macro')
best_model = classifier.get_best_model(metric='f1_macro')

train: LogisticRegression_OVR
done
Repetición 1, Modelo LogisticRegression_OVR completado.
train: LogisticRegression_OVO
done
Repetición 1, Modelo LogisticRegression_OVO completado.
train: RandomForest_OVO
done
Repetición 1, Modelo RandomForest_OVO completado.
train: RandomForest_OVR
done
Repetición 1, Modelo RandomForest_OVR completado.
train: XGBoost_OVR
done
Repetición 1, Modelo XGBoost_OVR completado.
train: XGBoost_OVO
done
Repetición 1, Modelo XGBoost_OVO completado.
train: KNN_OVR
done
Repetición 1, Modelo KNN_OVR completado.
train: DecisionTree_OVO
done
Repetición 1, Modelo DecisionTree_OVO completado.
train: DecisionTree_OVR
done
Repetición 1, Modelo DecisionTree_OVR completado.
train: NaiveBayes_OVR
done
Repetición 1, Modelo NaiveBayes_OVR completado.
train: NaiveBayes_OVO
done
Repetición 1, Modelo NaiveBayes_OVO completado.
train: GradientBoosting_OVR
done
Repetición 1, Modelo GradientBoosting_OVR completado.
train: GradientBoosting_OVO


In [ ]:
classifier2 = MultiClassClassifier(
    models=models,
    df=df,
    numeric_features=numeric_features,apply_pca=True,
    n_components = 2)

classifier2.train(X, y, n_repeats=30)


classifier2.plot_results(metrics=['f1_macro', 'f1_weighted', 'accuracy', 'precision_macro', 'recall_macro'])
classifier2.plot_mean_confusion_matrices(normalize='true')

stats_results = classifier2.perform_statistical_analysis(metric='f1_macro')
best_model = classifier2.get_best_model(metric='f1_macro')

train: LogisticRegression_OVR


In [ ]:
classifier1 = MultiClassClassifier(
    models=models,
    df=df,
    numeric_features=numeric_features,
    feature_selection=True,
    k_features=2
   )

classifier1.train(X, y, n_repeats=30)


classifier1.plot_results(metrics=['f1_macro', 'f1_weighted', 'accuracy', 'precision_macro', 'recall_macro'])
classifier1.plot_mean_confusion_matrices(normalize='true')

stats_results = classifier1.perform_statistical_analysis(metric='f1_macro')
best_model = classifier1.get_best_model(metric='f1_macro')